In [2]:
import helpers
from helpers import init_batch, generate_next_token
from helpers import merge_batches, filter_batch

In [3]:
import copy
import random
import matplotlib.pyplot as plt
import torch
import time
import torch.nn.functional as F
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

In [4]:
model_name = "D:/doc/model_weights/qwen3-0.5b"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [5]:
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id

tokenizer.pad_size = "left"
tokenizer.truncation_size = "left"

In [6]:
prompts = [
    "The quick brown fox jumped over the",
    "The rain in Spain falls",
    "What comes up must",
]

# note: padding=True ensures the padding token will be inserted into the tokenized tensors
inputs = tokenizer(prompts, padding=True, return_tensors="pt")

In [7]:
def generate_batch_tokens_with_past(inputs):
    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    last_logits = logits[:, -1, :]
    next_token_ids = last_logits.argmax(dim=1, keepdim=True)
    return next_token_ids, outputs.past_key_values


def generate_batch(inputs, max_tokens):
    # create a list of tokens for every input in the batch
    generated_tokens = [[] for _ in range(inputs["input_ids"].shape[0])]
    
    attention_mask = inputs["attention_mask"]
    position_ids = attention_mask.long().cumsum(-1) - 1
    position_ids.masked_fill_(attention_mask == 0, 1)
    
    next_inputs = {
        "position_ids": position_ids,
        **inputs
    }
    for _ in range(max_tokens):
        next_token_ids, past_key_values = generate_batch_tokens_with_past(next_inputs)
        next_inputs = {
            "input_ids": next_token_ids,  # '-1' here means the remaining elements for this dim
            "position_ids": next_inputs["position_ids"][:, -1].unsqueeze(-1) + 1,  # increment last, discard the rest
            "attention_mask": torch.cat([
                next_inputs["attention_mask"],
                torch.ones((next_token_ids.shape[0], 1)),  # concatenate vector of 1's with shape [batch_size]
            ], dim=1),
            "past_key_values": past_key_values,
        }

        next_tokens = tokenizer.batch_decode(next_token_ids)
        for i, token in enumerate(next_tokens):
            generated_tokens[i].append(token)
    return ["".join(tokens) for tokens in generated_tokens]

In [8]:
random.seed(42)
queue_size = 32
batch_size = 8

request_queue = [
    (prompts[0], 100 if i % batch_size == 0 else 10)
    for i in range(queue_size)
]

In [9]:
batches = [
    request_queue[i: i + batch_size]
    for i in range(0, len(request_queue), batch_size)
]
batches[:2]

[[('The quick brown fox jumped over the', 100),
  ('The quick brown fox jumped over the', 10),
  ('The quick brown fox jumped over the', 10),
  ('The quick brown fox jumped over the', 10),
  ('The quick brown fox jumped over the', 10),
  ('The quick brown fox jumped over the', 10),
  ('The quick brown fox jumped over the', 10),
  ('The quick brown fox jumped over the', 10)],
 [('The quick brown fox jumped over the', 100),
  ('The quick brown fox jumped over the', 10),
  ('The quick brown fox jumped over the', 10),
  ('The quick brown fox jumped over the', 10),
  ('The quick brown fox jumped over the', 10),
  ('The quick brown fox jumped over the', 10),
  ('The quick brown fox jumped over the', 10),
  ('The quick brown fox jumped over the', 10)]]

In [11]:
t0 = time.time()
with tqdm(total=len(batches), desc=f"bs={batch_size}") as pbar:
    for i, batch in enumerate(batches):
        batch_max_tokens = [b[1] for b in batch]
        max_tokens = max(batch_max_tokens)
        pbar.set_postfix({"max_tokens": max_tokens})

        batch_prompts = [b[0] for b in batch]
        inputs = tokenizer(
            batch_prompts, padding=True, return_tensors="pt"
        )

        generate_batch(inputs, max_tokens=max_tokens)
        pbar.update(1)

duration_s = time.time() - t0
print("duration_s:", duration_s)

bs=8: 100%|██████████| 4/4 [00:43<00:00, 10.94s/it, max_tokens=100]

duration_s: 43.769283056259155
